# FASTT — ablations (Table 5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/ablation.ipynb)

Two ablations, each holding everything fixed but one design choice.

* **5a — the feature transformation.** Per corpus and FASTT variant, the best hold-out corpus WER reached by each
  transformation (identity, diagonal, linear, low-rank, nonlinear) in the extended hyperparameter exploration on
  the corpus's main trio: configurations drawn from a discrete grid, each scored with three inner folds on the
  hold-out the main search uses. The trials are read from the Hub; nothing is fitted.
* **5b — the training objective (AMI-SDM).** The same architecture, features, search budget, partition and 5×2
  folds, trained once with cross-entropy on the per-clip best expert and once with the expected-WER objective,
  then compared fold by fold.

| Level | What runs | Needs |
|---|---|---|
| **1** | both tables, from the trials, searches and folds stored on the Hub | a CPU, minutes |
| **2** | 5b: the cross-entropy controls' folds refitted (`RETUNE = True`: their search as well) | a CPU, about an hour |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [2]:
from fastt.ablation import LossAblation, transform_cells, transform_table
from fastt.configs import MAIN

LEVEL = 1          # 1 or 2 (see above); Table 5a is read at every level
RETUNE = False     # level 2: run the cross-entropy controls' search again

## 1. The feature transformation (Table 5a)

Bold: the best transformation in the row. `*`: the transformation the main search selected under its budget of
ten configurations. Where the two differ, the main search's smaller budget landed on a transformation the richer
exploration ranks lower, which is why the table is read row by row.

In [3]:
cells = transform_cells()
transform_table(cells)

Table 5a — feature transformation: best hold-out corpus WER per transform
the extended exploration on each corpus's main trio; * = the transform the main search selected (budget of 10); bold = best in the row
  [VoxPopuli]
          variant    Raw  Diag. Linear Low-rank Nonlin.  trials
    FASTT-Boosted 0.1767 0.1770 0.1770  0.1774*  0.1773      40
FASTT-Alternating 0.1763 0.1762 0.1764   0.1774 0.1773*      39
  [AMI (SDM)]
          variant     Raw   Diag. Linear Low-rank Nonlin.  trials
    FASTT-Boosted 0.6184*  0.6243 0.6230   0.6244  0.6219      40
FASTT-Alternating  0.6228 0.6197* 0.6221   0.6218  0.6213      40
  [Earnings-22]
          variant     Raw   Diag. Linear Low-rank Nonlin.  trials
    FASTT-Boosted  0.4002 0.4187* 0.3992   0.3998  0.4003      20
FASTT-Alternating 0.4044*  0.4025 0.4010   0.4016  0.4022      20
  [People's Speech]
          variant     Raw  Diag.  Linear Low-rank Nonlin.  trials
    FASTT-Boosted 0.3908* 0.3925  0.3920   0.3935  0.3941      40
FASTT-Alternating  0.3912 0.3908 0.3907*   0.3909  0.3916      40

## 2. The training objective (Table 5b)

Each cross-entropy control is its expected-WER parent with one change: it is trained on the per-clip best expert
as a plain label, with no WER-derived weights. It is searched over its parent's space with the main run's budget,
sampler and partition, and scored on the main run's ten folds; the expected-WER rows are the main run's own.

In [4]:
abl = LossAblation(MAIN["ami_sdm"], level=LEVEL, retune=RETUNE)
abl.load()

AMI-SDM: level 1 — search read from the Hub, folds read from the Hub
experts  ESPnet Conformer-RNN-T, ESPnet Transformer, wav2vec2-large-960h
arms     FASTT-Alternating, FASTT-Boosted, Raw XGBoost (AdaSTT), Raw SDT, SelectKBest XGBoost, SelectKBest SDT, ROVER (weighted), ROVER (confidence-weighted), CN-MBR, Score-Fusion, Hard-Selector (plain CE), Hard-Selector (cost-sensitive CE), Soft-Selector (mixture), QE-Router


41,641 utterances x 3 routable model(s), from cache .fastt-cache/store_f06450c105e3398c
texts for 3 model(s), from cache
100.0% of 1,200 (utterance, model) counters reproduce exactly


EnsembleStore(ami_sdm, 'mean', 41,641 utterances, 3 routable)

In [5]:
abl.search();


=== best argmin_clf   corpus WER 0.6227   gap closed +24.1%   (10 trials, 0.0 min)
{
  "alpha": 0.1
}

=== best probe   corpus WER 0.6302   gap closed +15.8%   (10 trials, 0.0 min)
{
  "alpha": 100.0,
  "objective_weight": "length"
}

=== best qe_router   corpus WER 0.6288   gap closed +17.4%   (10 trials, 0.0 min)
{
  "alpha_decades": 4,
  "alpha_lo": 0.0001,
  "n_alphas": 16,
  "standardize": false,
  "target": "errors"
}

=== best adastt   corpus WER 0.6206   gap closed +26.5%   (10 trials, 0.0 min)
{
  "sel_colsample": 1.0,
  "sel_learning_rate": 0.1,
  "sel_max_depth": 3,
  "sel_min_child_weight": 16.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 0.3,
  "sel_subsample": 0.6
}

=== best fastt_boosted   corpus WER 0.6182   gap closed +29.2%   (10 trials, 0.0 min)
{
  "adam_lr": 0.0003,
  "adam_weight_decay": 0.001,
  "batch_size": 512,
  "depth": 7,
  "epochs": 200,
  "lmbda": 1.0,
  "lmbda2": 1e-07,
  "num_rounds": 2,
  "num_trees": 8,
  "patience": 20,
  "transform_type": "ide


=== best adastt_ce   corpus WER 0.6233   gap closed +23.5%   (10 trials, 0.0 min)
{
  "sel_colsample": 1.0,
  "sel_learning_rate": 0.1,
  "sel_max_depth": 3,
  "sel_min_child_weight": 16.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 0.3,
  "sel_subsample": 0.6
}

=== best fastt_boosted_ce   corpus WER 0.6225   gap closed +24.4%   (10 trials, 0.0 min)
{
  "adam_lr": 0.0003,
  "adam_weight_decay": 1e-07,
  "batch_size": 512,
  "bottleneck_ratio": 0.15,
  "depth": 7,
  "epochs": 200,
  "lmbda": 1e-06,
  "lmbda2": 1e-07,
  "num_rounds": 6,
  "num_trees": 24,
  "out_ratio": 0.1,
  "patience": 20,
  "t_weight_decay": 1e-06,
  "transform_type": "low_rank"
}

=== best fastt_alternating_ce   corpus WER 0.6241   gap closed +22.7%   (10 trials, 0.0 min)
{
  "bottleneck_ratio": 0.05,
  "num_iterations": 6,
  "out_ratio": 0.3,
  "sel_colsample": 0.1,
  "sel_learning_rate": 0.3,
  "sel_max_depth": 8,
  "sel_min_child_weight": 1.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 3.0,
  "sel_subsa

In [6]:
abl.cross_validate();

  runlog HIT  cv5x2 seed 11 — 10/10 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 22 — 10/10 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 33 — 10/10 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 44 — 10/10 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 55 — 10/10 group(s) cached, nothing refitted


The whole ladder, from linear selectors trained with plain and cost-sensitive cross-entropy up to the two FASTT
variants; then the three paired comparisons, with Alpaydın's combined 5×2 cv $F$ test.

In [7]:
abl.ladder()

arm,architecture,loss,corpus WER,mean utt. WER,GC@0 (%)
Hard-Selector (plain CE),linear,plain CE,0.6284 ± 0.0038,0.8771 ± 0.0026,25.9 ± 1.0
Hard-Selector (cost-sensitive CE),linear,cost-sensitive CE,0.6392 ± 0.0052,0.8830 ± 0.0032,14.4 ± 2.7
QE-Router,ridge regression,"per-expert WER, argmin",0.6272 ± 0.0025,0.9006 ± 0.0031,27.1 ± 0.9
"Raw XGBoost (AdaSTT), CE",XGBoost,plain CE,0.6313 ± 0.0034,0.8801 ± 0.0029,22.8 ± 0.8
Raw XGBoost (AdaSTT),XGBoost,expected WER,0.6274 ± 0.0029,0.8585 ± 0.0024,26.9 ± 1.6
"FASTT-Boosted, CE",FASTT-Boosted,plain CE,0.6280 ± 0.0039,0.8790 ± 0.0029,26.2 ± 2.4
FASTT-Boosted,FASTT-Boosted,expected WER,0.6221 ± 0.0030,0.8583 ± 0.0033,32.4 ± 1.1
"FASTT-Alternating, CE",FASTT-Alternating,plain CE (both swaps),0.6314 ± 0.0036,0.8759 ± 0.0024,22.6 ± 1.2
FASTT-Alternating,FASTT-Alternating,expected WER,0.6240 ± 0.0030,0.8611 ± 0.0031,30.5 ± 0.8


In [8]:
abl.tests()

architecture,CE,E[WER],Δ (points),p (F)
XGBoost,0.6313,0.6274,+0.39,0.033
FASTT-Boosted,0.6280,0.6221,+0.59,0.006
FASTT-Alternating,0.6314,0.6240,+0.74,<0.001


## 3. Record

Both ablations' tables and the controls' selected hyperparameters, in `ablation_ami_sdm.json`.

In [9]:
abl.save();

wrote ablation_ami_sdm.json
